# Lecture 13: Multiple Regression And Workflow Design

This notebook moves from a simple model to a multiple-regression workflow with explicit feature selection, validation, and interpretation steps.


In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

from pathlib import Path


def find_repo_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists():
            return path
    raise RuntimeError("Could not find repository root")


ROOT = find_repo_root()
DATA = ROOT / "data" / "raw"


In [ ]:
df = pd.read_csv(DATA / "career_outcomes.csv")
df.describe(include="all")


## Workflow

1. Define the prediction or explanation goal.
2. Split outcome and predictors.
3. Fit a transparent baseline.
4. Add predictors that answer a substantive question.
5. Evaluate whether the added complexity improves the model.


In [ ]:
simple = smf.ols("salary_k_eur ~ training_hours", data=df).fit()
multiple = smf.ols(
    "salary_k_eur ~ training_hours + experience_years + education_years + ai_tool_use",
    data=df,
).fit()

comparison = pd.DataFrame(
    {
        "model": ["simple", "multiple"],
        "r_squared": [simple.rsquared, multiple.rsquared],
        "adjusted_r_squared": [simple.rsquared_adj, multiple.rsquared_adj],
        "residual_std_error": [np.sqrt(simple.scale), np.sqrt(multiple.scale)],
    }
)
comparison


In [ ]:
print(multiple.summary())


In [ ]:
features = ["training_hours", "experience_years", "education_years", "ai_tool_use"]
X = df[features]
y = df["salary_k_eur"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
model = LinearRegression()
model.fit(X_train, y_train)
pred = model.predict(X_test)

rmse = mean_squared_error(y_test, pred) ** 0.5
r2 = r2_score(y_test, pred)
print(f"Test RMSE: {rmse:.2f}")
print(f"Test R-squared: {r2:.3f}")


In [ ]:
pd.Series(model.coef_, index=features).sort_values(key=abs, ascending=False)


## LLM Planning Exercise

Ask an LLM for a model-building plan. Mark each proposed step as one of: data preparation, modeling, validation, interpretation, or unsupported claim.
